In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import os
import yaml

In [2]:
#Load the configuration file
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

path = "../"+config["paths"]["raw_data_dir"]    

In [3]:

def get_parquet_from_path(path):
    parquet_files = []
    for (dir_path, _, files) in os.walk(path):
        for f in files:
            if f.endswith(".parquet"):
                parquet_files.append(os.path.join(dir_path, f)) 
    return parquet_files

In [4]:
file_path = get_parquet_from_path(path)
(print(file_path))


['../data/raw/cscareerquestionsEU_comments.parquet', '../data/raw/SoftwareEngineering_comments.parquet', '../data/raw/cscareerquestions_comments.parquet', '../data/raw/SoftwareEngineering_submissions.parquet', '../data/raw/datascience_submissions.parquet', '../data/raw/cybersecurity_submissions.parquet', '../data/raw/QualityAssurance_comments.parquet', '../data/raw/datascience_comments.parquet', '../data/raw/QualityAssurance_submissions.parquet', '../data/raw/cscareerquestionsEU_submissions.parquet', '../data/raw/cscareerquestions_submissions.parquet', '../data/raw/dataengineering_submissions.parquet', '../data/raw/cybersecurity_comments.parquet', '../data/raw/dataengineering_comments.parquet']


In [5]:
ts=pq.read_metadata(file_path[1])
ts.num_rows, ts.num_columns, pq.read_schema(file_path[0])

(41518,
 5,
 id: string
 subreddit: string
 created_utc: timestamp[ns]
 score: int64
 text: string
 -- schema metadata --
 pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 849)

In [26]:
import json
import random
def sample_parquet_file_to_json(file_path, desired_samples = 10000):
    file_count = 0
    for file_number in range(len(file_path)):
        file_name = os.path.splitext(os.path.basename(file_path[file_number]))[0]
        # Read the paraquet file metadata
        metadata = pq.read_metadata(file_path[file_number])
        # Total number of samples and target retention number
        total_samples = metadata.num_rows
        # Perform random sampling
        if total_samples <= desired_samples:
            table = pq.read_table(file_path[file_number])
            data = table.to_pylist()
        else:
            random_indices = random.sample(range(total_samples), desired_samples)
            random_indices.sort()  # Sort the indices to speed up batch reading
            table = pq.read_table(file_path[file_number])
          
            # Take a batch based on the random  indices
            sampled_table = table.take(random_indices)
            selected_samples = sampled_table.to_pylist()
        # Write the sampled results to a new JSON file
        
        with open(f'data_samples/{file_name}.json','w', encoding='utf-8') as f:
            json.dump(selected_samples, f, ensure_ascii=False, indent=4, default=str)
        print(f"Random sampling completed for {file_name}, retained {len(selected_samples)} samples.\n") 
        file_count=+1
    print("Done!")   

In [23]:
sample_parquet_file_to_json(file_path)

Random sampling completed for cscareerquestionsEU_comments, retained 10000 samples.
/n
Random sampling completed for SoftwareEngineering_comments, retained 10000 samples.
/n
Random sampling completed for cscareerquestions_comments, retained 10000 samples.
/n
Random sampling completed for SoftwareEngineering_submissions, retained 10000 samples.
/n
Random sampling completed for datascience_submissions, retained 10000 samples.
/n
Random sampling completed for cybersecurity_submissions, retained 10000 samples.
/n
Random sampling completed for QualityAssurance_comments, retained 10000 samples.
/n
Random sampling completed for datascience_comments, retained 10000 samples.
/n
Random sampling completed for QualityAssurance_submissions, retained 10000 samples.
/n
Random sampling completed for cscareerquestionsEU_submissions, retained 10000 samples.
/n
Random sampling completed for cscareerquestions_submissions, retained 10000 samples.
/n
Random sampling completed for dataengineering_submissions